# 08 — Census disparate-impact fairness audit (Chicago / NYC / LA)

- **Owner:** Bella  
- **Date:** 2026-07-12  
- **Models audited:** Model 1 (risk) — full battery; Model 2 (forecast) — calibration + coverage.  
- **Basis:** each city's chronological **test split** with realised labels (not the forward-looking `scores.json`).  
- **Artifacts written:** `reports/fairness/fairness_audit_<city>.json`.

> **The one rule:** census demographics are **audit-only** — they measure disparate impact and are never a model feature (decisions 0004 / 0005). This notebook joins them *after* the model, never before.

The audit reads a city-agnostic `AuditFrame` from a per-city adapter, joins ACS tract demographics (`foodsafety.audit.census`), and runs the metrics engine (`foodsafety.audit.fairness`): flag-rate parity, FPR, FNR, and calibration by group, each with bootstrap CIs and a material-and-confident verdict. See `src/foodsafety/audit/README.md` for the design.

Requires `CENSUS_API_KEY` in the environment and the `audit` extra (`uv sync --extra audit`).

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import average_precision_score

from foodsafety.audit import census, fairness, report, mitigation
from foodsafety.audit.adapters.chicago import ChicagoAdapter
from foodsafety.audit.adapters.nyc import NycAdapter
from foodsafety.audit.adapters.la import LaAdapter
from foodsafety.audit.census import ACS_YEAR

pd.set_option('display.width', 200, 'display.max_columns', 30)
OUT = Path('..') / 'reports' / 'fairness'
OUT.mkdir(parents=True, exist_ok=True)
ADAPTERS = {'chicago': ChicagoAdapter(), 'nyc': NycAdapter(), 'la': LaAdapter()}

## Build each city's audit frame, join census, and write the report
Each adapter reproduces its city's deployed model on the test split; the census join adds tract demographics; `report.build_report` assembles the reviewable JSON.

In [ ]:
reports = {}
for city, adapter in ADAPTERS.items():
    frame = adapter.build_audit_frame()
    frame = census.attach_area_demographics(frame, city=city)
    rep = report.build_report(frame, city, acs_year=ACS_YEAR)
    (OUT / f'fairness_audit_{city}.json').write_text(json.dumps(rep, indent=2))
    reports[city] = rep
    p = rep['provenance']
    print(f"{city:8s} rows={p['test_rows']:>6}  prevalence={p['label_prevalence']:.3f}  "
          f"M1 PR-AUC={p['model1_test_pr_auc']:.3f}  window={p['test_window']}")

## Per-city summary — which axes fire, on which lens
The key read: a **parity-only** finding (flag rate differs) is expected wherever true prevalence differs across groups — that is the model flagging higher-risk places, not bias. The lenses that catch bias are **FPR / FNR / calibration**, because they condition on the realised label.

In [ ]:
for city, rep in reports.items():
    print(f'==================== {city.upper()} ====================')
    print(pd.DataFrame(rep['model1_risk']['summary']).to_string(index=False))
    print()

## Verdict per axis (Model 1), with prevalence-tracking and the secondary operating point

In [ ]:
for city, rep in reports.items():
    print(f'-------------------- {city.upper()} --------------------')
    for key, ax in rep['model1_risk']['axes'].items():
        print(f"[{key}] corr={ax['flag_vs_prevalence_corr']}")
        print('   ', ax['verdict'])
    print()

## Model 2 (forecast) — calibration by group
The forecast has no flagging operating point, so only calibration + coverage are audited.

In [ ]:
for city, rep in reports.items():
    rows = [{'axis': k, 'coverage': a['coverage'], 'ece_gap': a['ece_gap'],
             'finding': a['finding']} for k, a in rep['model2_forecast']['axes'].items()]
    print(f'{city.upper()}:')
    print(pd.DataFrame(rows).to_string(index=False))
    print()

## Mitigation cost (analysis only)
If an equalized-odds gap appeared, this prices the per-group thresholds that equalize recall. It does **not** change the model — adopting per-group thresholds is a scope call (Jun).

In [ ]:
chi = census.attach_area_demographics(ADAPTERS['chicago'].build_audit_frame(), city='chicago')
tbl = mitigation.equalize_recall_thresholds(chi, 'area_income_q')
print(tbl.to_string(index=False))
print('extra inspections to equalize recall across income quartiles:',
      tbl.attrs['total_delta_flags'], 'of', tbl.attrs['baseline_flagged'], 'flagged')